### Import thư viện và Thiết lập môi trường

In [ ]:
import os
import json
import math
import random
import numpy as np
from pathlib import Path
from PIL import Image
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import GradScaler, autocast  # Mixed precision
 
import torchvision.transforms as T
import torchvision.models as models
 
import albumentations as A
from albumentations.pytorch import ToTensorV2
 
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    confusion_matrix, classification_report
)


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\anhng\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\anhng\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Error importing huggingface_hub.hf_api: No module named 'httpx'


### Config 

In [ ]:
class CFG:
    # --- Roboflow ---
    ROBOFLOW_API_KEY  = "m84yy3vIbXagdMl6aOZd"   # ← thay key của bạn
    WORKSPACE         = "hi-anh"       # ← thay workspace
    PROJECT           = "bok-choy-lettuce-spinach-diseased-jg4ui"    # ← thay project name
    VERSION           = 2                      # ← version dataset
 
    # --- Dataset ---
    DATA_DIR          = Path("/kaggle/working/dataset")
    IMG_SIZE          = 224
    NUM_CLASSES       = 1                      # Binary: disease / no-disease
 
    # --- Training ---
    EPOCHS            = 40
    BATCH_SIZE        = 32
    LR_MAX            = 3e-4
    WEIGHT_DECAY      = 1e-4
    POS_WEIGHT        = 2.0    # Tăng nếu dataset mất cân bằng (ít ảnh bệnh hơn)
 
    # --- Model ---
    BACKBONE          = "resnet50"             # resnet18 | resnet34 | resnet50
    PRETRAINED        = True
    DROPOUT           = 0.4
    FREEZE_EPOCHS     = 5                      # Số epoch đóng băng backbone
 
    # --- Augmentation ---
    MIXUP_ALPHA       = 0.3
 
    # --- Paths ---
    CKPT_DIR          = Path("/kaggle/working/checkpoints")
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    BEST_CKPT         = CKPT_DIR / "best_model.pth"
 
    # --- Misc ---
    SEED              = 42
    NUM_WORKERS       = 2
    DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
print(f"✅ Device: {CFG.DEVICE}")
print(f"✅ Backbone: {CFG.BACKBONE} | Pretrained: {CFG.PRETRAINED}")
 

✅ Device: cuda
✅ Backbone: resnet50 | Pretrained: True


### Seed & Reproducibility

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
 
set_seed(CFG.SEED)

### Tải dữ liệu từ Roboflow

In [ ]:
def download_dataset():
    """
    Tải dataset từ Roboflow về Kaggle working directory.
    Format: YOLOv8 (object detection)
        dataset/
          train/
            images/   ← ảnh
            labels/   ← file .txt (class cx cy w h)
          valid/...
          test/...
    """
    from roboflow import Roboflow

    rf      = Roboflow(api_key=CFG.ROBOFLOW_API_KEY)
    project = rf.workspace(CFG.WORKSPACE).project(CFG.PROJECT)
    dataset = project.version(CFG.VERSION).download(
        "yolov8",
        location=str(CFG.DATA_DIR),
        overwrite=True
    )
    print(f"✅ Dataset tải về: {CFG.DATA_DIR}")
    return dataset


def inspect_dataset():
    """In cấu trúc thư mục YOLO và thống kê label để kiểm tra."""
    print("\n📁 Cấu trúc dataset (YOLO format):")
    for split in ("train", "valid", "test"):
        split_dir = CFG.DATA_DIR / split
        if not split_dir.exists():
            print(f"  [{split:5s}] ❌ không tồn tại")
            continue

        img_dir = split_dir / "images"
        lbl_dir = split_dir / "labels"

        imgs   = list(img_dir.glob("*.*")) if img_dir.exists() else []
        labels = list(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []

        # Đếm ảnh có annotation (label file không rỗng) vs không có (healthy)
        n_disease = sum(1 for f in labels if f.stat().st_size > 0)
        n_healthy = len(imgs) - n_disease

        print(f"  [{split:5s}] {len(imgs):5d} ảnh | "
              f"{n_disease:4d} disease | {n_healthy:4d} healthy "
              f"(positive rate: {n_disease/max(len(imgs),1):.1%})")


### Dataset Class

In [ ]:
class PlantDiseaseDataset(Dataset):
    """
    Dataset cho YOLO format (1 class: Disease).

    Cấu trúc thư mục:
        <split>/
          images/  ← ảnh (.jpg/.png)
          labels/  ← file .txt tương ứng
                      - Tồn tại & không rỗng  → label = 1 (disease)
                      - Không tồn tại / rỗng  → label = 0 (healthy)

    Logic gán nhãn:
      Roboflow YOLO chỉ tạo file .txt khi có bounding box.
      Ảnh không có annotation → không có file label → label = 0.
    """

    def __init__(self, root_dir, split="train", transform=None):
        self.transform   = transform
        self.image_paths = []
        self.labels      = []

        img_dir = Path(root_dir) / split / "images"
        lbl_dir = Path(root_dir) / split / "labels"

        if not img_dir.exists():
            raise FileNotFoundError(f"Không tìm thấy thư mục ảnh: {img_dir}")

        for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG"):
            for img_path in sorted(img_dir.glob(ext)):
                # Tìm file label tương ứng (cùng tên, đổi đuôi .txt)
                lbl_path = lbl_dir / (img_path.stem + ".txt")

                # label = 1 nếu file label tồn tại và không rỗng
                if lbl_path.exists() and lbl_path.stat().st_size > 0:
                    label = 1.0
                else:
                    label = 0.0

                self.image_paths.append(img_path)
                self.labels.append(label)

        n_total   = len(self.labels)
        n_disease = int(sum(self.labels))
        n_healthy = n_total - n_disease
        print(f"[{split:5s}] {n_total} ảnh | "
              f"Disease: {n_disease} | Healthy: {n_healthy} | "
              f"Positive rate: {n_disease/max(n_total,1):.2%}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img   = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        label = self.labels[idx]

        if self.transform:
            img = self.transform(image=img)["image"]

        return img, torch.tensor(label, dtype=torch.float32)


### Augmentation

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
 
def get_transforms(split="train"):
    """
    Albumentations >= 2.x thay đổi API so với 1.x:
      ✅ RandomResizedCrop : size=(H,W) tuple  (không phải 2 số riêng)
      ✅ GaussNoise        : std_range=(lo,hi)  (không phải var_limit)
      ✅ CoarseDropout     : num_holes_range, hole_height_range, hole_width_range
      ✅ Rotate            : thay RandomRotation
    """
    s = CFG.IMG_SIZE
    if split == "train":
        return A.Compose([
            A.RandomResizedCrop(size=(s, s), scale=(0.6, 1.0), p=1.0),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),          # Lá cây không có chiều cố định
            A.Rotate(limit=45, p=0.5),
            A.OneOf([
                A.ColorJitter(brightness=0.3, contrast=0.3,
                              saturation=0.3, hue=0.1),
                A.HueSaturationValue(hue_shift_limit=20,
                                     sat_shift_limit=40,
                                     val_shift_limit=30),
            ], p=0.7),
            A.OneOf([
                A.GaussNoise(std_range=(0.02, 0.1)),  # std_range thay var_limit
                A.GaussianBlur(blur_limit=(3, 7)),
                A.MotionBlur(blur_limit=5),
            ], p=0.3),
            A.RandomBrightnessContrast(p=0.3),
            A.CLAHE(clip_limit=4.0, p=0.2),
            A.CoarseDropout(                          # API mới albumentations 2.x
                num_holes_range=(1, 8),
                hole_height_range=(16, 32),
                hole_width_range=(16, 32),
                fill=0, p=0.3
            ),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(s, s),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])

### Mixup Augmentation

In [ ]:
def mixup_data(x, y, alpha=0.3):
    """Trộn 2 ảnh ngẫu nhiên → giúp model học boundary mượt hơn."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index       = torch.randperm(batch_size, device=x.device)
    mixed_x     = lam * x + (1 - lam) * x[index]
    y_a, y_b    = y, y[index]
    return mixed_x, y_a, y_b, lam
 
 
def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

### Mô hình

In [ ]:
class ChannelAttention(nn.Module):
    """Channel Attention từ CBAM — nhấn mạnh các feature map quan trọng."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()
 
    def forward(self, x):
        return x * self.sigmoid(self.fc(self.avg_pool(x)) +
                                self.fc(self.max_pool(x)))
 
 
class PlantDiseaseResNet(nn.Module):
    """
    Kiến trúc:
      ResNet backbone (pretrained ImageNet)
        └─ Thay layer4 cuối + thêm CBAM attention
        └─ Head: GAP → Dropout → Linear(1) → BCEWithLogits
    
    Lý do chọn ResNet:
      - Residual connections tránh vanishing gradient
      - Pretrained weights trên ImageNet cho feature extraction tốt
      - Dễ fine-tune với dataset nhỏ (~8k ảnh)
    """
    def __init__(self, backbone="resnet50", pretrained=True, dropout=0.4):
        super().__init__()
 
        # --- Load backbone ---
        weights = "IMAGENET1K_V2" if pretrained else None
        if backbone == "resnet18":
            base = models.resnet18(weights=weights)
            feat_dim = 512
        elif backbone == "resnet34":
            base = models.resnet34(weights=weights)
            feat_dim = 512
        elif backbone == "resnet50":
            base = models.resnet50(weights="IMAGENET1K_V2" if pretrained else None)
            feat_dim = 2048
        else:
            raise ValueError(f"Backbone không hợp lệ: {backbone}")
 
        # --- Tách backbone thành các stages ---
        self.stage0 = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.stage1 = base.layer1   # stride 4
        self.stage2 = base.layer2   # stride 8
        self.stage3 = base.layer3   # stride 16
        self.stage4 = base.layer4   # stride 32
 
        # --- CBAM Attention sau stage4 ---
        self.attention = ChannelAttention(feat_dim)
 
        # --- Head ---
        self.gap       = nn.AdaptiveAvgPool2d(1)
        self.dropout   = nn.Dropout(p=dropout)
        self.classifier = nn.Linear(feat_dim, 1)  # Binary logit
 
        # --- Khởi tạo head ---
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)
 
    def freeze_backbone(self):
        """Đóng băng toàn bộ backbone, chỉ train head."""
        for param in [*self.stage0.parameters(),
                      *self.stage1.parameters(),
                      *self.stage2.parameters(),
                      *self.stage3.parameters(),
                      *self.stage4.parameters()]:
            param.requires_grad = False
        print("🔒 Backbone FROZEN")
 
    def unfreeze_backbone(self):
        """Mở khóa toàn bộ backbone để fine-tune."""
        for param in self.parameters():
            param.requires_grad = True
        print("🔓 Backbone UNFROZEN — Fine-tuning toàn bộ mạng")
 
    def forward(self, x):
        x = self.stage0(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
 
        x = self.attention(x)       # Channel attention
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.classifier(x)   # Raw logit

### Metrics

In [ ]:
class MetricTracker:
    def __init__(self):
        self.reset()
 
    def reset(self):
        self.preds  = []
        self.labels = []
        self.losses = []
 
    def update(self, loss, logits, labels):
        self.losses.append(loss)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        self.preds.extend(probs.flatten().tolist())
        self.labels.extend(labels.detach().cpu().numpy().flatten().tolist())
 
    def compute(self):
        avg_loss = np.mean(self.losses)
        preds_np  = np.array(self.preds)
        labels_np = np.array(self.labels)
 
        # Binary predictions với ngưỡng 0.5
        bin_preds = (preds_np >= 0.5).astype(int)
        acc       = (bin_preds == labels_np).mean()
 
        # Average Precision (AP) — không phụ thuộc threshold
        if len(np.unique(labels_np)) > 1:
            ap = average_precision_score(labels_np, preds_np)
        else:
            ap = float("nan")
 
        # F1
        tp = ((bin_preds == 1) & (labels_np == 1)).sum()
        fp = ((bin_preds == 1) & (labels_np == 0)).sum()
        fn = ((bin_preds == 0) & (labels_np == 1)).sum()
        f1 = (2 * tp) / (2 * tp + fp + fn + 1e-8)
 
        return {"loss": avg_loss, "acc": acc, "ap": ap, "f1": f1}

### Training loop

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion,
                    scaler, scheduler, epoch):
    model.train()
    tracker = MetricTracker()
 
    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(CFG.DEVICE)
        labels = labels.to(CFG.DEVICE)
 
        # MixUp augmentation
        images, y_a, y_b, lam = mixup_data(images, labels, CFG.MIXUP_ALPHA)
 
        optimizer.zero_grad()
 
        # Mixed Precision Forward
        with autocast():
            logits = model(images).squeeze(1)
            loss   = mixup_criterion(criterion, logits, y_a, y_b, lam)
 
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
 
        tracker.update(loss.item(), logits, labels)
 
        if batch_idx % 20 == 0:
            print(f"  Epoch {epoch} | Step {batch_idx}/{len(loader)} "
                  f"| Loss: {loss.item():.4f} "
                  f"| LR: {scheduler.get_last_lr()[0]:.2e}")
 
    return tracker.compute()
 
 
@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    tracker = MetricTracker()
 
    for images, labels in loader:
        images = images.to(CFG.DEVICE)
        labels = labels.to(CFG.DEVICE)
 
        with autocast():
            logits = model(images).squeeze(1)
            loss   = criterion(logits, labels)
 
        tracker.update(loss.item(), logits, labels)
 
    return tracker.compute()

In [ ]:
def train(model, train_loader, val_loader):
    # Loss function có pos_weight để xử lý mất cân bằng
    pos_weight = torch.tensor([CFG.POS_WEIGHT], device=CFG.DEVICE)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    scaler  = GradScaler()
    history = {"train": [], "val": []}
    best_ap = 0.0

    # ── Giai đoạn 1: Freeze backbone, chỉ train head ────────────────────────
    print(f"\n🔒 Giai đoạn 1: Freeze backbone ({CFG.FREEZE_EPOCHS} epochs)")
    model.freeze_backbone()

    optimizer_head = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CFG.LR_MAX * 10, weight_decay=CFG.WEIGHT_DECAY
    )
    scheduler_head = OneCycleLR(
        optimizer_head, max_lr=CFG.LR_MAX * 10,
        total_steps=CFG.FREEZE_EPOCHS * len(train_loader),
        pct_start=0.2, anneal_strategy="cos"
    )

    for epoch in range(1, CFG.FREEZE_EPOCHS + 1):
        train_metrics = train_one_epoch(
            model, train_loader, optimizer_head, criterion,
            scaler, scheduler_head, epoch
        )
        val_metrics = validate(model, val_loader, criterion)
        history["train"].append(train_metrics)
        history["val"].append(val_metrics)
        _print_epoch(epoch, CFG.EPOCHS, train_metrics, val_metrics)
        best_ap = _save_if_best(model, epoch, val_metrics, best_ap)

    # ── Giai đoạn 2: Unfreeze, fine-tune toàn bộ ────────────────────────────
    remaining = CFG.EPOCHS - CFG.FREEZE_EPOCHS
    print(f"\n🔓 Giai đoạn 2: Fine-tune toàn bộ ({remaining} epochs)")
    model.unfreeze_backbone()

    optimizer_full = torch.optim.AdamW(
        model.parameters(),
        lr=CFG.LR_MAX, weight_decay=CFG.WEIGHT_DECAY
    )
    scheduler_full = OneCycleLR(
        optimizer_full, max_lr=CFG.LR_MAX,
        total_steps=remaining * len(train_loader),
        pct_start=0.05, anneal_strategy="cos"
    )

    for epoch in range(CFG.FREEZE_EPOCHS + 1, CFG.EPOCHS + 1):
        train_metrics = train_one_epoch(
            model, train_loader, optimizer_full, criterion,
            scaler, scheduler_full, epoch
        )
        val_metrics = validate(model, val_loader, criterion)
        history["train"].append(train_metrics)
        history["val"].append(val_metrics)
        _print_epoch(epoch, CFG.EPOCHS, train_metrics, val_metrics)
        best_ap = _save_if_best(model, epoch, val_metrics, best_ap)

    return history


def _print_epoch(epoch, total, train_m, val_m):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch:3d}/{total}")
    print(f"  TRAIN | Loss:{train_m['loss']:.4f} Acc:{train_m['acc']:.4f} "
          f"AP:{train_m['ap']:.4f} F1:{train_m['f1']:.4f}")
    print(f"  VAL   | Loss:{val_m['loss']:.4f} Acc:{val_m['acc']:.4f} "
          f"AP:{val_m['ap']:.4f} F1:{val_m['f1']:.4f}")
    print(f"{'='*60}\n")


def _save_if_best(model, epoch, val_metrics, best_ap):
    ap = val_metrics["ap"]
    if not math.isnan(ap) and ap > best_ap:
        torch.save({
            "epoch":       epoch,
            "state_dict":  model.state_dict(),
            "best_ap":     ap,
            "val_metrics": val_metrics,
            "config": {
                "backbone":    CFG.BACKBONE,
                "img_size":    CFG.IMG_SIZE,
                "num_classes": CFG.NUM_CLASSES,
            }
        }, CFG.BEST_CKPT)
        print(f"  💾 Saved best checkpoint (AP={ap:.4f})")
        return ap
    return best_ap


### Visualization

In [ ]:
def plot_history(history):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics = ["loss", "acc", "ap"]
    titles  = ["Loss", "Accuracy", "Average Precision"]
 
    for ax, metric, title in zip(axes, metrics, titles):
        train_vals = [m[metric] for m in history["train"]
                      if not math.isnan(m[metric])]
        val_vals   = [m[metric] for m in history["val"]
                      if not math.isnan(m[metric])]
 
        ax.plot(train_vals, label="Train", color="#2196F3", linewidth=2)
        ax.plot(val_vals,   label="Val",   color="#F44336", linewidth=2)
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(alpha=0.3)
 
    plt.suptitle("PlantBot Training History", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig("/kaggle/working/training_history.png", dpi=150)
    plt.show()
    print("✅ Lưu biểu đồ: training_history.png")
 
 
@torch.no_grad()
def evaluate_on_test(model, test_loader, threshold=0.5):
    """Đánh giá chi tiết trên test set."""
    model.eval()
    all_preds  = []
    all_labels = []
 
    for images, labels in test_loader:
        images = images.to(CFG.DEVICE)
        logits = model(images).squeeze(1)
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_preds.extend(probs.flatten().tolist())
        all_labels.extend(labels.numpy().flatten().tolist())
 
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    bin_preds  = (all_preds >= threshold).astype(int)
 
    print("\n" + "="*50)
    print("📊 ĐÁNH GIÁ TRÊN TEST SET")
    print("="*50)
 
    if len(np.unique(all_labels)) > 1:
        print(f"Average Precision (AP) : {average_precision_score(all_labels, all_preds):.4f}")
        print(f"\nClassification Report:")
        print(classification_report(
            all_labels, bin_preds,
            target_names=["Healthy", "Disease"]
        ))
 
        # Precision-Recall Curve
        precision, recall, thresholds = precision_recall_curve(all_labels, all_preds)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
 
        ax1.plot(recall, precision, color="#4CAF50", linewidth=2)
        ax1.fill_between(recall, precision, alpha=0.1, color="#4CAF50")
        ax1.set_xlabel("Recall")
        ax1.set_ylabel("Precision")
        ax1.set_title("Precision-Recall Curve")
        ax1.grid(alpha=0.3)
 
        # F1 theo threshold
        f1_scores = 2 * precision[:-1] * recall[:-1] / (
            precision[:-1] + recall[:-1] + 1e-8
        )
        best_thresh_idx = np.argmax(f1_scores)
        ax2.plot(thresholds, f1_scores, color="#9C27B0", linewidth=2)
        ax2.axvline(thresholds[best_thresh_idx], color="red",
                    linestyle="--", label=f"Best={thresholds[best_thresh_idx]:.2f}")
        ax2.set_xlabel("Threshold")
        ax2.set_ylabel("F1 Score")
        ax2.set_title("F1 vs Threshold")
        ax2.legend()
        ax2.grid(alpha=0.3)
 
        plt.tight_layout()
        plt.savefig("/kaggle/working/evaluation.png", dpi=150)
        plt.show()
        print(f"✅ Ngưỡng tốt nhất: {thresholds[best_thresh_idx]:.3f} "
              f"(F1={f1_scores[best_thresh_idx]:.4f})")
    else:
        print("⚠️  Test set chỉ có 1 class — không tính AP được.")
        print(f"Accuracy: {(bin_preds == all_labels).mean():.4f}")

In [ ]:
def predict_image(model, image_path, threshold=0.5):
    """
    Dự đoán 1 ảnh bất kỳ.
    Args:
        image_path: str hoặc Path — đường dẫn tới file ảnh
        threshold : ngưỡng phân loại (mặc định 0.5)
    Returns:
        prob  (float) : xác suất disease [0, 1]
        label (str)   : "🦠 DISEASE" hoặc "✅ HEALTHY"
    """
    model.eval()
    transform = get_transforms("val")

    img = np.array(Image.open(image_path).convert("RGB"))
    inp = transform(image=img)["image"].unsqueeze(0).to(CFG.DEVICE)

    with torch.no_grad():
        logit = model(inp).squeeze()
        prob  = torch.sigmoid(logit).item()

    label = "🦠 DISEASE" if prob >= threshold else "✅ HEALTHY"
    return prob, label


# ── MAIN ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # --- Bước 1: Tải dataset ---
    print("\n📥 TẢI DATASET TỪ ROBOFLOW...")
    download_dataset()

    # --- Bước 2: Kiểm tra cấu trúc ---
    inspect_dataset()

    # --- Bước 3: Tạo DataLoaders ---
    print("\n📂 CHUẨN BỊ DATALOADER...")
    train_dataset = PlantDiseaseDataset(CFG.DATA_DIR, split="train",
                                        transform=get_transforms("train"))
    val_dataset   = PlantDiseaseDataset(CFG.DATA_DIR, split="valid",
                                        transform=get_transforms("val"))
    test_dataset  = PlantDiseaseDataset(CFG.DATA_DIR, split="test",
                                        transform=get_transforms("val"))

    # Tính pos_weight từ dữ liệu thực tế thay vì hardcode
    n_pos = int(sum(train_dataset.labels))
    n_neg = len(train_dataset.labels) - n_pos
    CFG.POS_WEIGHT = n_neg / max(n_pos, 1)
    print(f"  ⚖️  POS_WEIGHT tự động: {CFG.POS_WEIGHT:.2f} "
          f"(neg={n_neg}, pos={n_pos})")

    train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE,
                              shuffle=True,  num_workers=CFG.NUM_WORKERS,
                              pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_dataset,   batch_size=CFG.BATCH_SIZE,
                              shuffle=False, num_workers=CFG.NUM_WORKERS,
                              pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=CFG.BATCH_SIZE,
                              shuffle=False, num_workers=CFG.NUM_WORKERS,
                              pin_memory=True)

    # --- Bước 4: Khởi tạo mô hình ---
    print("\n🏗️  KHỞI TẠO MÔ HÌNH...")
    model = PlantDiseaseResNet(
        backbone=CFG.BACKBONE,
        pretrained=CFG.PRETRAINED,
        dropout=CFG.DROPOUT
    ).to(CFG.DEVICE)

    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Tổng tham số    : {total_params:,}")
    print(f"  Tham số trainable: {trainable_params:,}")

    # --- Bước 5: Huấn luyện ---
    print("\n🚀 BẮT ĐẦU HUẤN LUYỆN...")
    history = train(model, train_loader, val_loader)

    # --- Bước 6: Vẽ biểu đồ ---
    plot_history(history)

    # --- Bước 7: Load best model & đánh giá test set ---
    print("\n🔄 Load best checkpoint...")
    ckpt = torch.load(CFG.BEST_CKPT, map_location=CFG.DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    print(f"  Best epoch: {ckpt['epoch']} | Val AP: {ckpt['best_ap']:.4f}")

    evaluate_on_test(model, test_loader)

    # --- Bước 8: Inference ví dụ ---
    final_ckpt_path = CFG.CKPT_DIR / "final_model.pth"
    torch.save({
        "epoch":       CFG.EPOCHS,
        "state_dict":  model.state_dict(),
        "config": {
            "backbone":    CFG.BACKBONE,
            "img_size":    CFG.IMG_SIZE,
            "num_classes": CFG.NUM_CLASSES,
            "dropout":     CFG.DROPOUT,
        }
    }, final_ckpt_path)
    print(f"💾 Lưu final model: {final_ckpt_path}")

    # (Tuỳ chọn) Load lại để kiểm tra
    ckpt  = torch.load(final_ckpt_path, map_location=CFG.DEVICE)
    model_reload = PlantDiseaseResNet(
        backbone=ckpt["config"]["backbone"],
        pretrained=False,
        dropout=ckpt["config"]["dropout"]
    ).to(CFG.DEVICE)
    model_reload.load_state_dict(ckpt["state_dict"])
    model_reload.eval()
    print("✅ Load lại thành công — sẵn sàng inference")
    # sample_path = CFG.DATA_DIR / "test" / "images" / "your_image.jpg"
    # prob, label = predict_image(model, sample_path)
    # print(f"\nKết quả: {label} (prob={prob:.3f})")



📥 TẢI DATASET TỪ ROBOFLOW...


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to \kaggle\working\dataset in yolov8:: 100%|██████████| 7155/7155 [00:04<00:00, 1537.35it/s]


✅ Dataset tải về: \kaggle\working\dataset

📂 CHUẨN BỊ DATALOADER...
[train] 5006 ảnh | Classes: ['images', 'labels'] | Positive rate: 0.00%
[valid] 1430 ảnh | Classes: ['images', 'labels'] | Positive rate: 0.00%
[test] 714 ảnh | Classes: ['images', 'labels'] | Positive rate: 0.00%

🏗️  KHỞI TẠO MÔ HÌNH...
  Tổng tham số:    24,034,369
  Tham số trainable: 24,034,369

🚀 BẮT ĐẦU HUẤN LUYỆN...
🔒 Backbone FROZEN


`torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
